[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C3/notebooks/vakciny.ipynb)

# Rozdělení vakcín — samostatná úloha

**Pracujete sami.** Model si napíšete z textu zadání, vyřešíte ho v CVXPY
a z multiplikátorů vyčtete, co by přinesla další dodávka a proč jeden okres
nedostal nic.

## Zadání slovy

> Kraj dostal **30 tisíc dávek** vakcíny pro rizikové skupiny ve čtyřech okresech:
> Město, Průmysl, Venkov a Hory. Epidemiologický model odhaduje, že když okres $i$
> dostane $x_i$ tisíc dávek, odvrátí se tam za sezónu $a_i\ln(1+x_i)$ hospitalizací.
> Logaritmus vyjadřuje **klesající výnos**: první tisícovky jdou k nejohroženějším
> a pomůžou nejvíc, každá další o něco méně.
>
> | okres | koeficient $a_i$ | riziková populace [tis. osob] |
> |---|---:|---:|
> | Město | 60 | 10 |
> | Průmysl | 40 | 12 |
> | Venkov | 30 | 18 |
> | Hory | 3 | 6 |
>
> Víc dávek, než má okres rizikových lidí, mu nedáme (každý dostane nejvýš jednu).
> Dávky se nedají brát, takže záporné množství nedává smysl. Všechno počítejte
> v **tisících dávek**.
>
> **Jak dávky rozdělit, aby se odvrátilo co nejvíc hospitalizací?**

## Postup

1. Napište model na papír: proměnné, cíl, omezení. Pak ho doplňte v CVXPY *(díra 1)*.
2. Co přinese tisícovka dávek navíc? Vyčtěte multiplikátor dodávky $\mu_D$ a ověřte přepočtem *(díra 2)*.
3. Kdo je na stropu, kdo uvnitř a kdo na nule, a proč?
4. Bonus pro rychlé: každý okres dostane aspoň tisíc dávek *(díra 3)*.

---

*Čísla, se kterými se notebook porovnává, počítá skript [`kod/vakciny.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C3/kod/vakciny.py).*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

OKRESY = ["Město", "Průmysl", "Venkov", "Hory"]
A = np.array([60.0, 40.0, 30.0, 3.0])    # x tisíc dávek odvrátí a * ln(1 + x) hospitalizací
N = np.array([10.0, 12.0, 18.0, 6.0])    # riziková populace [tis. osob] = strop
CELKEM = 30.0                            # dodávka [tis. dávek]

## Krok 1 — model v CVXPY *(díra 1)*

In [ ]:
# TODO 1: doplňte pojmenovaná omezení a úlohu
x = cp.Variable(4)                 # tisíce dávek pro Město, Průmysl, Venkov, Hory
dodavka = ...                      # celkem nejvýš CELKEM tisíc dávek
strop = ...                        # nejvýš riziková populace N
nula = ...                         # nezápornost
omezeni = [dodavka, strop, nula]
uloha = cp.Problem(..., omezeni)   # cp.Maximize, nebo cp.Minimize? nápověda: A @ cp.log(...)

In [ ]:
uloha.solve(solver=cp.CLARABEL)
# výsledky si uložíme hned: další řešení (krok 2) přepíše x.value i multiplikátory
x_opt, odvraceno = x.value.copy(), uloha.value
mu_D = float(dodavka.dual_value)          # cena dodávky: hospitalizace na tisíc dávek navíc
mu_strop, mu_nula = strop.dual_value.copy(), nula.dual_value.copy()

print(f"odvráceno {odvraceno:.2f} hospitalizací, mu_D = {mu_D:.3f}")
for okres, xi in zip(OKRESY, x_opt):
    print(f"  {okres:8s} x = {xi:6.2f} tis.")
assert np.allclose(x_opt, [10, 11.571, 8.429, 0], atol=1e-2) and abs(odvraceno - 312.44) < 1e-2

## Krok 2 — tisícovka navíc a mezní přínosy *(díra 2)*

Multiplikátor $\mu_D$ odhaduje, o kolik vzroste optimum, když dodávka vzroste o tisíc
dávek. Ověřte to přepočtem: sestavte stejnou úlohu s dodávkou 31 tisíc. Pak porovnejte
mezní přínos $a_i/(1+x_i)$ každého okresu s $\mu_D$.

In [ ]:
# TODO 2: mezní přínos další tisícovky v každém okrese a stejná úloha pro 31 tis.
mezni = ...                                   # a_i / (1 + x_i) pro všechny okresy najednou (numpy, x_opt)
uloha_31 = cp.Problem(uloha.objective, [...])  # dodávka 31 místo CELKEM, strop a nula beze změny

In [ ]:
prirustek_31 = uloha_31.solve(solver=cp.CLARABEL) - odvraceno
print(f"31 tis.: skutečně +{prirustek_31:.2f}, odhad mu_D = {mu_D:.2f}")
for okres, m in zip(OKRESY, mezni):
    print(f"  {okres:8s} mezní přínos {m:5.2f}   (mu_D = {mu_D:.2f})")
assert np.allclose(mezni, [5.4545, 3.1818, 3.1818, 3.0], atol=1e-2) and abs(prirustek_31 - 3.11) < 1e-2

h = np.linspace(0, 18, 300)
for i, okres in enumerate(OKRESY):
    hi = h[h <= N[i]]
    plt.plot(hi, A[i] / (1 + hi), label=okres)
    plt.plot(x_opt[i], mezni[i], "o", color=f"C{i}")
plt.axhline(mu_D, color="k", ls="--", label=f"hladina mu_D = {mu_D:.2f}")
plt.ylim(0, 12); plt.xlabel("dávky v okrese [tis.]"); plt.ylabel("mezní přínos a/(1+x)")
plt.legend(); plt.show()

**Vodní hladina.** Každý okres dostává dávky, dokud jeho mezní přínos neklesne
na společnou „hladinu“ $\mu_D$ (čárkovaná čára v grafu). Stacionarita z KKT po složkách
říká: mezní přínos = cena dodávky + cena stropu − cena nuly,

$$
\frac{a_i}{1+x_i} = \mu_D + \mu_i^{\text{strop}} - \mu_i^{\text{nula}} .
$$

**TODO:** Kdo je na stropu, kdo uvnitř a kdo na nule, a proč? Rozhodněte z grafu
a spočítejte multiplikátory kalkulačkou. Pak spusťte buňku níže a porovnejte.

In [ ]:
print(f"mu_D = {mu_D:.2f}")
for okres, m, ms, mn in zip(OKRESY, mezni, mu_strop, mu_nula):
    print(f"  {okres:8s} mezní přínos {m:5.2f}   mu_strop = {ms:5.2f}   mu_nula = {mn:5.2f}")

## Bonus — spravedlnost *(díra 3)*

Hory nedostanou nic. Je to spravedlivé? Zkusme, aby každý okres dostal aspoň tisíc
dávek. Kolik hospitalizací tahle spravedlnost stojí?

In [ ]:
# TODO 3 (bonus): stejná úloha, ale každý okres dostane aspoň 1 tisíc dávek.
x_s = cp.Variable(4)
spravedliva = cp.Problem(..., [...])   # stejná účelová funkce jako v díře 1, ale pro x_s

In [ ]:
spravedliva.solve(solver=cp.CLARABEL)
print(f"x = {np.round(x_s.value, 2)}, odvráceno {spravedliva.value:.2f}, "
      f"spravedlnost stojí {odvraceno - spravedliva.value:.2f} hospitalizace")
assert abs(spravedliva.value - 311.27) < 1e-2